# AndinaLog 03B — Notebook 2: tratamiento IoT didáctico

Este notebook recibe el **CSV diagnosticado didáctico** del Notebook 1. No necesita catálogo ni motor externos. Conserva las columnas originales y sus estados/motivos; agrega valores tratados y explica las decisiones.

**Salidas:** un CSV de lecturas utilizables y otro de cuarentena final. El CSV de cuarentena del Notebook 1 es un subconjunto del diagnosticado y no se concatena, para evitar duplicar filas.

**Política:** la hora sin zona se interpreta como `America/La_Paz` y se deriva UTC. Celsius es la unidad canónica; Fahrenheit se convierte. Kelvin permanece en cuarentena. `-999` es faltante. La imputación solo se permite entre dos lecturas cercanas, válidas y coherentes del mismo viaje, camión y producto. Nunca se imputan identificadores, fechas o banderas. Las lecturas imputadas no son observaciones reales para KPIs de excursión ni para crear una etiqueta de predicción.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
MAX_INTERVALO_VECINOS_MIN = 60
MAX_CAMBIO_TEMPERATURA_C = 2.0
MAX_CAMBIO_HUMEDAD_PCT = 20.0
COLUMNAS_BRONZE = ["timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag"]

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas/andinalog_iot_telemetry_didactico_v1_diagnosticado.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas/andinalog_iot_telemetry_didactico_v1_diagnosticado.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
ENTRADA = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas/andinalog_iot_telemetry_didactico_v1_diagnosticado.csv"
SALIDAS = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_iot_telemetry/salidas"
df = pd.read_csv(ENTRADA, dtype="string", encoding="utf-8-sig", keep_default_na=False)
requeridas = ["fila_bronze", "en_cuarentena", "motivos_fila", "zona_horaria_origen", *COLUMNAS_BRONZE]
requeridas += [f"{c}_{s}" for c in COLUMNAS_BRONZE for s in ("estado", "motivo")]
faltantes = sorted(set(requeridas) - set(df.columns))
if faltantes: raise ValueError(f"Faltan columnas del diagnóstico: {faltantes}")
if df["fila_bronze"].duplicated().any(): raise ValueError("fila_bronze debe ser único")
if not df["zona_horaria_origen"].eq("America/La_Paz").all():
    raise ValueError("La zona de origen debe ser America/La_Paz")
if not df["en_cuarentena"].isin(["True", "False"]).all():
    raise ValueError("en_cuarentena debe ser True o False")
original = df[requeridas].copy(deep=True)
print("Filas diagnosticadas:", len(df))


Filas diagnosticadas: 28920


## 1. Correcciones deterministas

Las columnas `*_tratado` son nuevas. Las originales y las marcas del diagnóstico no se sobrescriben. Los motivos y acciones del tratamiento se agregan a nivel de fila.


In [2]:
df["acciones_tratamiento"] = ""
df["motivos_tratamiento"] = ""
def anotar(mascara, accion, motivo=""):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    for campo, texto in [("acciones_tratamiento", accion), ("motivos_tratamiento", motivo)]:
        if texto:
            previo = df.loc[mascara, campo]
            df.loc[mascara, campo] = previo.where(previo.eq(""), previo + " | ") + texto

df["camion_id_tratado"] = df["camion_id"].str.strip().str.upper()
camion_normalizado = df["camion_id_tratado"].ne(df["camion_id"]) & df["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False)
anotar(camion_normalizado, "NORMALIZAR_CAMION_ID", "Mayúsculas y espacios normalizados")

df["temp_unit_tratado"] = df["temp_unit"].str.strip().str.upper()
temp_raw = pd.to_numeric(df["temperatura_cabina_c"], errors="coerce")
centinela = temp_raw.eq(-999)
temp_raw = temp_raw.mask(centinela)
df["temperatura_cabina_c_tratada"] = temp_raw.where(df["temp_unit_tratado"].eq("C"),
    (temp_raw - 32) * 5 / 9).where(df["temp_unit_tratado"].isin(["C", "F"]))
convertida = df["temp_unit_tratado"].eq("F") & df["temperatura_cabina_c_tratada"].notna()
anotar(convertida, "CONVERTIR_F_A_C", "Fahrenheit convertido a Celsius")
anotar(centinela, "CENTINELA_A_FALTANTE", "-999 no representa temperatura")
df["temp_unit_tratado"] = df["temp_unit_tratado"].mask(df["temp_unit_tratado"].isin(["C", "F"]), "C")

humedad_raw = pd.to_numeric(df["humedad_cabina_pct"], errors="coerce")
df["humedad_cabina_pct_tratada"] = humedad_raw.where(humedad_raw.between(0, 100))

fecha_local = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
df["timestamp_utc"] = fecha_local.dt.tz_localize("America/La_Paz").dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ").fillna("")
anotar(fecha_local.notna(), "DERIVAR_UTC", "Timestamp local de Bolivia convertido a UTC")


## 2. Imputación restringida

Se interpolan faltantes interiores solamente. Los dos vecinos deben ser **observaciones válidas**, del mismo `viaje_id`, `camion_id_tratado` y `producto_id`, separados por hasta 60 minutos, con banderas térmicas `0` y sin duplicidad de clave. Se exige además una variación máxima de 2 °C o 20 puntos de humedad entre vecinos. Esos límites son parámetros conservadores del ejercicio y deben revisarse con el responsable del sensor. No se interpola sobre una excursión ni se usa una fila ya imputada como vecina.


In [3]:
# Duplicados exactos y conflictos de clave se deciden con el Bronze original.
firma = pd.util.hash_pandas_object(df[COLUMNAS_BRONZE], index=False)
clave = [df["viaje_id"], df["timestamp"]]
variantes = firma.groupby(clave, dropna=False).transform("nunique")
clave_repetida = df.duplicated(["viaje_id", "timestamp"], keep=False)
copia = clave_repetida & variantes.eq(1) & df.duplicated(COLUMNAS_BRONZE, keep="first")
conflicto = clave_repetida & variantes.gt(1)
anotar(copia, "EXCLUIR_COPIA", "Copia exacta de otra lectura")
anotar(conflicto, "CUARENTENA_CONFLICTO", "Misma clave de lectura con datos distintos")

df["temperatura_imputada"] = False
df["humedad_imputada"] = False
grupos = ["viaje_id", "camion_id_tratado", "producto_id"]
orden = df.assign(_fecha=fecha_local).sort_values(grupos + ["_fecha", "fila_bronze"], kind="stable")
orden = orden.loc[orden["_fecha"].notna()].copy()
g = orden.groupby(grupos, sort=False, dropna=False)
anterior = g["_fecha"].shift(1)
siguiente = g["_fecha"].shift(-1)
duracion = (siguiente - anterior).dt.total_seconds() / 60
interior = anterior.lt(orden["_fecha"]) & orden["_fecha"].lt(siguiente) & duracion.le(MAX_INTERVALO_VECINOS_MIN)
sin_conflicto = ~(copia | conflicto)
vecinos_sin_conflicto = sin_conflicto.reindex(orden.index).groupby([orden[c] for c in grupos], dropna=False).shift(1).fillna(False).astype(bool) & sin_conflicto.reindex(orden.index).groupby([orden[c] for c in grupos], dropna=False).shift(-1).fillna(False).astype(bool)
flags_cero = orden["desviacion_termica_flag"].eq("0") & orden["desviacion_proximos_60min_flag"].eq("0")
vecinos_flags_cero = flags_cero.groupby([orden[c] for c in grupos], dropna=False).shift(1).fillna(False).astype(bool) & flags_cero.groupby([orden[c] for c in grupos], dropna=False).shift(-1).fillna(False).astype(bool)
base = interior & sin_conflicto.reindex(orden.index) & vecinos_sin_conflicto & flags_cero & vecinos_flags_cero
fraccion = (orden["_fecha"] - anterior).dt.total_seconds() / (siguiente - anterior).dt.total_seconds()

def interpolar(columna, max_cambio, marca, accion):
    valor = pd.to_numeric(orden[columna], errors="coerce")
    antes = valor.groupby([orden[c] for c in grupos], dropna=False).shift(1)
    despues = valor.groupby([orden[c] for c in grupos], dropna=False).shift(-1)
    elegible = base & valor.isna() & antes.notna() & despues.notna() & (despues - antes).abs().le(max_cambio)
    if columna.startswith("temperatura"):
        unidad_observada = orden["temp_unit"].str.strip().str.upper()
        unidades_vecinas = unidad_observada.groupby([orden[c] for c in grupos], dropna=False)
        elegible &= unidad_observada.eq("C") & unidades_vecinas.shift(1).isin(["C", "F"]) & unidades_vecinas.shift(-1).isin(["C", "F"])
    nuevos = antes + fraccion * (despues - antes)
    indices = orden.index[elegible.fillna(False)]
    df.loc[indices, columna] = nuevos.loc[indices].round(3)
    df.loc[indices, marca] = True
    anotar(df.index.isin(indices), accion, "Interpolación entre dos lecturas observadas cercanas y coherentes")

interpolar("temperatura_cabina_c_tratada", MAX_CAMBIO_TEMPERATURA_C, "temperatura_imputada", "IMPUTAR_TEMPERATURA")
interpolar("humedad_cabina_pct_tratada", MAX_CAMBIO_HUMEDAD_PCT, "humedad_imputada", "IMPUTAR_HUMEDAD")


## 3. Decisión final y uso analítico

Una fila recuperada puede pasar a datos tratados aunque el diagnóstico original dijera cuarentena; ambas decisiones quedan disponibles. Kelvin, fecha inválida, identificador indispensable inválido, conflicto de clave o magnitud sin recuperación van a cuarentena final. Las advertencias de cobertura parcial de tablas maestras se conservan como advertencias.

`apta_kpi_termico` excluye temperaturas imputadas, unidades no válidas, productos sin umbral verificable y banderas inconsistentes. `apta_objetivo_60min` excluye filas con temperatura imputada para evitar utilizarla como observación de entrenamiento.


In [4]:
formato_ids = (
    df["viaje_id"].str.fullmatch(r"VIA-\d{5}").fillna(False) &
    df["order_id"].str.fullmatch(r"ORD-\d{4}-\d{5}").fillna(False) &
    df["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False) &
    df["producto_id"].str.fullmatch(r"PROD-\d{3}").fillna(False))
flags_validos = df["desviacion_termica_flag"].isin(["0", "1"]) & df["desviacion_proximos_60min_flag"].isin(["0", "1"])
unidad_valida = df["temp_unit"].str.strip().str.upper().isin(["C", "F"])
df["motivo_cuarentena_final"] = ""
def cuarentena_si(mascara, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(True).astype(bool)
    previo = df.loc[mascara, "motivo_cuarentena_final"]
    df.loc[mascara, "motivo_cuarentena_final"] = previo.where(previo.eq(""), previo + " | ") + motivo

cuarentena_si(fecha_local.isna(), "Fecha local inválida")
cuarentena_si(~formato_ids, "Identificador indispensable inválido")
cuarentena_si(~flags_validos, "Bandera inválida")
cuarentena_si(~unidad_valida, "Unidad térmica no operacional o desconocida")
cuarentena_si(df["temperatura_cabina_c_tratada"].isna(), "Temperatura sin valor recuperable")
cuarentena_si(df["humedad_cabina_pct_tratada"].isna(), "Humedad sin valor recuperable")
cuarentena_si(copia, "Copia exacta excluida")
cuarentena_si(conflicto, "Clave de lectura en conflicto")
df["en_cuarentena_final"] = df["motivo_cuarentena_final"].ne("")
df["decision_tratamiento"] = np.where(df["en_cuarentena_final"], "CUARENTENA", "TRATADO")

# La dimensión Productos Silver tiene cobertura parcial: ausencia implica NO_EVALUABLE, no ID inválido.
ruta_productos = RAIZ / "proyecto-integrador/andinalog_productos/notebook2/salidas/andinalog_productos_silver.csv"
df["umbral_producto_evaluable"] = False
df["flag_termico_coherente"] = False
if ruta_productos.is_file():
    productos = pd.read_csv(ruta_productos, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    if productos["producto_id"].duplicated().any(): raise ValueError("producto_id duplicado en Productos Silver")
    p = productos.set_index("producto_id")
    objetivo = pd.to_numeric(df["producto_id"].map(p["temperatura_conservacion_requerida_c"]), errors="coerce")
    tolerancia = pd.to_numeric(df["producto_id"].map(p["tolerancia_temperatura_c"]), errors="coerce")
    df["umbral_producto_evaluable"] = objetivo.notna() & tolerancia.ge(0)
    fuera = (pd.to_numeric(df["temperatura_cabina_c_tratada"], errors="coerce") - objetivo).abs().gt(tolerancia)
    df["flag_termico_coherente"] = df["umbral_producto_evaluable"] & fuera.eq(df["desviacion_termica_flag"].eq("1"))

df["apta_kpi_termico"] = (~df["en_cuarentena_final"] & ~df["temperatura_imputada"] &
    df["umbral_producto_evaluable"] & df["flag_termico_coherente"])
df["apta_objetivo_60min"] = ~df["en_cuarentena_final"] & ~df["temperatura_imputada"] & flags_validos
tratado = df.loc[~df["en_cuarentena_final"]].copy()
cuarentena_final = df.loc[df["en_cuarentena_final"]].copy()


## 4. Comprobaciones y exportación

Se exportan solo dos CSV de esta versión. Las filas en cuarentena siguen mostrando los valores tratados que fue posible derivar, pero no se mezclan con la salida utilizable.


In [5]:
pd.testing.assert_frame_equal(df[requeridas], original)
assert df["fila_bronze"].is_unique
assert len(df) == len(tratado) + len(cuarentena_final)
assert cuarentena_final["motivo_cuarentena_final"].ne("").all()
assert tratado["motivo_cuarentena_final"].eq("").all()
assert not (tratado["apta_kpi_termico"] & tratado["temperatura_imputada"]).any()
assert not (tratado["apta_objetivo_60min"] & tratado["temperatura_imputada"]).any()
assert not tratado["timestamp_utc"].eq("").any()
assert tratado["temp_unit_tratado"].eq("C").all()
assert not tratado["temperatura_cabina_c_tratada"].isna().any()
assert not tratado["humedad_cabina_pct_tratada"].isna().any()
assert not tratado["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False).eq(False).any()

SALIDAS.mkdir(parents=True, exist_ok=True)
ruta_tratado = SALIDAS / "andinalog_iot_telemetry_didactico_v1_tratado.csv"
ruta_cuarentena = SALIDAS / "andinalog_iot_telemetry_didactico_v1_cuarentena_final.csv"
tratado.to_csv(ruta_tratado, index=False, encoding="utf-8-sig")
cuarentena_final.to_csv(ruta_cuarentena, index=False, encoding="utf-8-sig")
print("Diagnosticadas:", len(df), "| Tratadas:", len(tratado), "| Cuarentena final:", len(cuarentena_final))
print("F a C:", int(convertida.sum()), "| Camiones normalizados:", int(camion_normalizado.sum()),
      "| Temperaturas imputadas:", int(df["temperatura_imputada"].sum()),
      "| Humedades imputadas:", int(df["humedad_imputada"].sum()))
print("Tratado:", ruta_tratado)
print("Cuarentena:", ruta_cuarentena)
display(df[["fila_bronze", "en_cuarentena", "decision_tratamiento", "acciones_tratamiento",
    "motivo_cuarentena_final", "apta_kpi_termico"]].head(10))


Diagnosticadas: 28920 | Tratadas: 28677 | Cuarentena final: 243
F a C: 50 | Camiones normalizados: 50 | Temperaturas imputadas: 120 | Humedades imputadas: 92
Tratado: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\02_tratamiento\andinalog_iot_telemetry\salidas\andinalog_iot_telemetry_didactico_v1_tratado.csv
Cuarentena: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\02_tratamiento\andinalog_iot_telemetry\salidas\andinalog_iot_telemetry_didactico_v1_cuarentena_final.csv


,fila_bronze,en_cuarentena,decision_tratamiento,acciones_tratamiento,motivo_cuarentena_final,apta_kpi_termico
0,1,False,TRATADO,DERIVAR_UTC,,True
1,2,False,TRATADO,DERIVAR_UTC,,True
2,3,False,TRATADO,DERIVAR_UTC,,True
3,4,False,TRATADO,DERIVAR_UTC,,True
4,5,False,TRATADO,DERIVAR_UTC,,True
5,6,False,TRATADO,DERIVAR_UTC,,True
6,7,False,TRATADO,DERIVAR_UTC,,True
7,8,True,TRATADO,DERIVAR_UTC | IMPUTAR_HUMEDAD,,True
8,9,False,TRATADO,DERIVAR_UTC,,True
9,10,False,TRATADO,DERIVAR_UTC,,True
